# MOGA-Phonons dispersion, mass, and lattice-parameter correlations

This clean notebook loads the top MOGA solutions from `dataframe000013.pkl` through `dataframe000016.pkl`, selects the top-$K$ overall solutions for each `(dataset, mass, lattice parameter)` group, and generates publication-quality figures exploring how the maximum phonon frequency varies with mass and lattice parameter.

The primary harmonic expectation is

\[
\omega \propto m^{-1/2},
\]

so the notebook emphasizes individual top-solution points rather than error bars or linear fits.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# User settings
# -----------------------------------------------------------------------------
PKL_DIR = Path("./dataframes/")
DATAFRAME_FILES = {
    "dataframe000013": PKL_DIR / "dataframe000013.pkl",
    "dataframe000014": PKL_DIR / "dataframe000014.pkl",
    "dataframe000015": PKL_DIR / "dataframe000015.pkl",
    "dataframe000016": PKL_DIR / "dataframe000016.pkl",
}

TOP_K = 5
RANK_BY = "fitness_norm"
OUTPUT_DIR = Path("mass_lattice_dispersion_publication_figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 12345

# -----------------------------------------------------------------------------
# Publication plotting defaults
# -----------------------------------------------------------------------------
mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "font.size": 18,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 14,
    "legend.title_fontsize": 14,
    "axes.linewidth": 1.8,
    "xtick.major.width": 1.6,
    "ytick.major.width": 1.6,
    "xtick.minor.width": 1.2,
    "ytick.minor.width": 1.2,
    "xtick.major.size": 7,
    "ytick.major.size": 7,
    "xtick.minor.size": 4,
    "ytick.minor.size": 4,
    "mathtext.default": "it",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def save_figure(fig, stem, output_dir=OUTPUT_DIR):
    """Save a figure as PDF, PNG, and SVG."""
    for ext in ["pdf", "png", "svg"]:
        fig.savefig(output_dir / f"{stem}.{ext}", bbox_inches="tight", dpi=600)


def find_first_existing(candidates, columns, required=True):
    """Return the first matching column name from a list of candidates."""
    for c in candidates:
        if c in columns:
            return c
    if required:
        raise KeyError(f"None of the candidate columns were found: {candidates}")
    return None


def label_from_dataframe_name(name):
    """Create a compact dataset label such as 013 from dataframe000013."""
    match = re.search(r"(\d+)$", str(name))
    return match.group(1)[-3:] if match else str(name)


## Load data and select top solutions

The notebook is intentionally defensive about column names because previous dataframe versions used slightly different labels. The selected dataframe `df_top` contains the top-$K$ entries ranked by `fitness_norm` for each combination of dataset, mass, and lattice parameter.


In [ ]:
frames = []
for label, path in DATAFRAME_FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Update PKL_DIR or DATAFRAME_FILES above."
        )
    tmp = pd.read_pickle(path).copy()
    tmp["dataset"] = label
    tmp["dataset_short"] = label_from_dataframe_name(label)
    frames.append(tmp)

df_all = pd.concat(frames, ignore_index=True)

mass_col = find_first_existing(["mass", "m"], df_all.columns)
alat_col = find_first_existing(["a_val", "alat", "a_latt", "lattice_parameter", "a"], df_all.columns)
rank_col = find_first_existing([RANK_BY, "fitness_norm", "fnorm"], df_all.columns)
max_freq_col = find_first_existing(["max_frequency", "max_freq", "omega_max", "omega_max_THz"], df_all.columns)
min_freq_col = find_first_existing(["min_frequency", "min_freq", "omega_min", "omega_min_THz"], df_all.columns, required=False)

# Convert key columns to numeric.
for col in [mass_col, alat_col, rank_col, max_freq_col, min_freq_col]:
    if col is not None:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

# Select top K overall solutions for each dataset, mass, and lattice parameter.
df_top = (
    df_all
    .dropna(subset=[mass_col, alat_col, rank_col, max_freq_col])
    .sort_values(rank_col, ascending=False)
    .groupby(["dataset", mass_col, alat_col], group_keys=False)
    .head(TOP_K)
    .copy()
)

df_top["omega_max_THz"] = df_top[max_freq_col]
df_top["mass_inv_sqrt"] = 1.0 / np.sqrt(df_top[mass_col])
df_top["omega_max_scaled"] = df_top["omega_max_THz"] * np.sqrt(df_top[mass_col])

if min_freq_col is not None:
    df_top["omega_min_THz"] = df_top[min_freq_col]

# Useful derived first-neighbor coordinates if force constants are available.
if {"alpha1", "beta1"}.issubset(df_top.columns):
    df_top["alpha1_plus_2beta1"] = df_top["alpha1"] + 2.0 * df_top["beta1"]
    df_top["alpha1_minus_beta1"] = df_top["alpha1"] - df_top["beta1"]

print(f"Loaded rows: {len(df_all):,}")
print(f"Top-{TOP_K} rows: {len(df_top):,}")
print(f"Mass column: {mass_col}")
print(f"Lattice-parameter column: {alat_col}")
print(f"Rank column: {rank_col}")
print(f"Maximum-frequency column: {max_freq_col}")
if min_freq_col is not None:
    print(f"Minimum-frequency column: {min_freq_col}")

display(
    df_top[["dataset", mass_col, alat_col, rank_col, "omega_max_THz", "mass_inv_sqrt", "omega_max_scaled"]]
    .head(10)
)


## Summary table

The table below is useful for checking how many top solutions are present at each mass and lattice parameter.


In [ ]:
summary = (
    df_top
    .groupby([alat_col, mass_col])
    .agg(
        omega_max_mean=("omega_max_THz", "mean"),
        omega_max_std=("omega_max_THz", "std"),
        omega_max_min=("omega_max_THz", "min"),
        omega_max_max=("omega_max_THz", "max"),
        n=("omega_max_THz", "size"),
    )
    .reset_index()
)
summary["omega_max_std"] = summary["omega_max_std"].fillna(0.0)
summary["mass_inv_sqrt"] = 1.0 / np.sqrt(summary[mass_col])
summary["omega_max_scaled_mean"] = summary["omega_max_mean"] * np.sqrt(summary[mass_col])

display(summary.head(15))
summary.to_csv(OUTPUT_DIR / "omega_max_mass_lattice_summary.csv", index=False)


## Figure 1 — Maximum phonon frequency versus inverse square-root mass

Individual top-solution points are shown with small horizontal jitter for visibility. The solid lines connect the mean value at each mass for fixed lattice parameter. No linear fit is imposed.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
fig, ax = plt.subplots(figsize=(8.0, 6.4))

unique_a = np.array(sorted(df_top[alat_col].unique()), dtype=float)
cmap = plt.get_cmap("viridis")
norm = mpl.colors.Normalize(vmin=unique_a.min(), vmax=unique_a.max())

for a_val in unique_a:
    sub = df_top[np.isclose(df_top[alat_col], a_val)].copy()
    sub = sub.sort_values("mass_inv_sqrt")
    color = cmap(norm(a_val))

    jitter = rng.normal(loc=0.0, scale=4.0e-4, size=len(sub))
    ax.scatter(
        sub["mass_inv_sqrt"] + jitter,
        sub["omega_max_THz"],
        s=36,
        color=color,
        edgecolor="black",
        linewidth=0.35,
        alpha=0.45,
        rasterized=True,
    )

    mean_sub = (
        sub.groupby(mass_col)
        .agg(omega_max_mean=("omega_max_THz", "mean"))
        .reset_index()
    )
    mean_sub["mass_inv_sqrt"] = 1.0 / np.sqrt(mean_sub[mass_col])
    mean_sub = mean_sub.sort_values("mass_inv_sqrt")

    ax.plot(
        mean_sub["mass_inv_sqrt"],
        mean_sub["omega_max_mean"],
        "-o",
        color=color,
        linewidth=2.4,
        markersize=6,
        markeredgecolor="black",
        markeredgewidth=0.5,
        label=fr"${a_val:g}$",
    )

ax.set_xlabel(r"$m^{-1/2}$ (amu$^{-1/2}$)")
ax.set_ylabel(r"$\omega_{\max}$ (THz)")
ax.set_title(r"Mass scaling of the maximum phonon frequency")
ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
ax.minorticks_on()

leg = ax.legend(
    frameon=False,
    title=r"$a$ (\AA)",
    bbox_to_anchor=(1.02, 0.5),
    loc="center left",
    handlelength=1.4,
)

fig.tight_layout()
save_figure(fig, "omega_max_vs_mass_inv_sqrt_individual_points")
plt.show()


## Figure 2 — Maximum phonon frequency versus mass, colored by lattice parameter

This is the same relationship plotted against mass directly. The decrease with increasing mass is the expected harmonic trend.


In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 6.4))

for a_val in unique_a:
    sub = df_top[np.isclose(df_top[alat_col], a_val)].copy()
    sub = sub.sort_values(mass_col)
    color = cmap(norm(a_val))

    jitter = rng.normal(loc=0.0, scale=0.16, size=len(sub))
    ax.scatter(
        sub[mass_col] + jitter,
        sub["omega_max_THz"],
        s=36,
        color=color,
        edgecolor="black",
        linewidth=0.35,
        alpha=0.45,
        rasterized=True,
    )

    mean_sub = (
        sub.groupby(mass_col)
        .agg(omega_max_mean=("omega_max_THz", "mean"))
        .reset_index()
        .sort_values(mass_col)
    )
    ax.plot(
        mean_sub[mass_col],
        mean_sub["omega_max_mean"],
        "-o",
        color=color,
        linewidth=2.4,
        markersize=6,
        markeredgecolor="black",
        markeredgewidth=0.5,
        label=fr"${a_val:g}$",
    )

ax.set_xlabel(r"$m$ (amu)")
ax.set_ylabel(r"$\omega_{\max}$ (THz)")
ax.set_title(r"Maximum phonon frequency decreases with mass")
ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
ax.minorticks_on()
ax.legend(frameon=False, title=r"$a$ (\AA)", bbox_to_anchor=(1.02, 0.5), loc="center left")

fig.tight_layout()
save_figure(fig, "omega_max_vs_mass_colored_by_lattice_parameter")
plt.show()


## Figure 3 — Mass-scaled maximum frequency versus lattice parameter

If simple mass scaling dominates, the quantity

\[
\omega_{\max}\sqrt{m}
\]

should largely remove the mass dependence. Residual variation with lattice parameter then reflects changes in effective force-constant scale.


In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 6.4))

unique_m = np.array(sorted(df_top[mass_col].unique()), dtype=float)
cmap_m = plt.get_cmap("plasma")
norm_m = mpl.colors.Normalize(vmin=unique_m.min(), vmax=unique_m.max())

for m_val in unique_m:
    sub = df_top[np.isclose(df_top[mass_col], m_val)].copy()
    sub = sub.sort_values(alat_col)
    color = cmap_m(norm_m(m_val))

    jitter = rng.normal(loc=0.0, scale=0.006, size=len(sub))
    ax.scatter(
        sub[alat_col] + jitter,
        sub["omega_max_scaled"],
        s=36,
        color=color,
        edgecolor="black",
        linewidth=0.35,
        alpha=0.45,
        rasterized=True,
    )

    mean_sub = (
        sub.groupby(alat_col)
        .agg(omega_max_scaled_mean=("omega_max_scaled", "mean"))
        .reset_index()
        .sort_values(alat_col)
    )
    ax.plot(
        mean_sub[alat_col],
        mean_sub["omega_max_scaled_mean"],
        "-o",
        color=color,
        linewidth=2.4,
        markersize=6,
        markeredgecolor="black",
        markeredgewidth=0.5,
        label=fr"${m_val:g}$",
    )

ax.set_xlabel(r"$a$ (\AA)")
ax.set_ylabel(r"$\omega_{\max}\sqrt{m}$ (THz amu$^{1/2}$)")
ax.set_title(r"Residual lattice-parameter dependence after mass scaling")
ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
ax.minorticks_on()
ax.legend(frameon=False, title=r"$m$ (amu)", bbox_to_anchor=(1.02, 0.5), loc="center left")

fig.tight_layout()
save_figure(fig, "omega_max_sqrt_mass_vs_lattice_parameter")
plt.show()


## Figure 4 — Heat map of mean maximum frequency

This heat map summarizes the mean maximum phonon frequency over the selected top solutions.


In [ ]:
pivot = summary.pivot(index=mass_col, columns=alat_col, values="omega_max_mean")
pivot = pivot.sort_index(ascending=True).sort_index(axis=1, ascending=True)

fig, ax = plt.subplots(figsize=(8.4, 6.4))
im = ax.imshow(
    pivot.values,
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    cmap="viridis",
)

ax.set_xticks(np.arange(pivot.shape[1]))
ax.set_yticks(np.arange(pivot.shape[0]))
ax.set_xticklabels([f"{x:g}" for x in pivot.columns])
ax.set_yticklabels([f"{y:g}" for y in pivot.index])
ax.set_xlabel(r"$a$ (\AA)")
ax.set_ylabel(r"$m$ (amu)")
ax.set_title(r"Mean $\omega_{\max}$ for top MOGA solutions")

cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label(r"$\langle \omega_{\max}\rangle$ (THz)")

ax.tick_params(axis="both", which="both", direction="out", top=False, right=False)

fig.tight_layout()
save_figure(fig, "omega_max_mean_heatmap_mass_lattice")
plt.show()


## Figure 5 — Correlation summary

This heat map reports Pearson correlations among the key scalar quantities in the selected top-solution set.


In [ ]:
corr_columns = [mass_col, alat_col, "mass_inv_sqrt", "omega_max_THz", "omega_max_scaled", rank_col]
if min_freq_col is not None:
    corr_columns.insert(4, "omega_min_THz")
for optional in ["fitness1", "fitness2", "fitness3", "num_imaginary", "alpha1_plus_2beta1", "alpha1_minus_beta1", "alpha2", "beta2"]:
    if optional in df_top.columns and optional not in corr_columns:
        corr_columns.append(optional)

corr_df = df_top[corr_columns].apply(pd.to_numeric, errors="coerce")
corr = corr_df.corr(method="pearson")
corr.to_csv(OUTPUT_DIR / "pearson_correlation_mass_lattice_dispersion.csv")

fig, ax = plt.subplots(figsize=(0.70 * len(corr.columns) + 4, 0.70 * len(corr.columns) + 3))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="coolwarm", interpolation="nearest")

labels = [
    r"$m$" if c == mass_col else
    r"$a$" if c == alat_col else
    r"$m^{-1/2}$" if c == "mass_inv_sqrt" else
    r"$\omega_{\max}$" if c == "omega_max_THz" else
    r"$\omega_{\min}$" if c == "omega_min_THz" else
    r"$\omega_{\max}\sqrt{m}$" if c == "omega_max_scaled" else
    r"$f_{\rm norm}$" if c == rank_col else
    r"$f_1$" if c == "fitness1" else
    r"$f_2$" if c == "fitness2" else
    r"$f_3$" if c == "fitness3" else
    r"$N_-$" if c == "num_imaginary" else
    r"$\alpha_1+2\beta_1$" if c == "alpha1_plus_2beta1" else
    r"$\alpha_1-\beta_1$" if c == "alpha1_minus_beta1" else
    r"$\alpha_2$" if c == "alpha2" else
    r"$\beta_2$" if c == "beta2" else
    str(c)
    for c in corr.columns
]

ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
ax.set_title("Pearson correlation matrix")

for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        val = corr.values[i, j]
        txt_color = "white" if abs(val) > 0.55 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=10, color=txt_color)

cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label(r"Pearson $r$")
ax.tick_params(axis="both", which="both", length=0)

fig.tight_layout()
save_figure(fig, "pearson_correlation_mass_lattice_dispersion")
plt.show()


## Figure 6 — Dataset reproducibility check

This figure checks whether the four independent dataframes give consistent mass-scaling trends.


In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 6.4))
markers = {"013": "o", "014": "s", "015": "^", "016": "D"}

# Color by lattice parameter, shape by dataset.
for label, sub_ds in df_top.groupby("dataset_short"):
    for a_val in unique_a:
        sub = sub_ds[np.isclose(sub_ds[alat_col], a_val)]
        if sub.empty:
            continue
        color = cmap(norm(a_val))
        ax.scatter(
            sub["mass_inv_sqrt"],
            sub["omega_max_THz"],
            s=34,
            marker=markers.get(label, "o"),
            color=color,
            edgecolor="black",
            linewidth=0.35,
            alpha=0.50,
            rasterized=True,
        )

# Legend for dataset markers only.
handles = [
    mpl.lines.Line2D([], [], marker=markers.get(label, "o"), linestyle="None", color="white",
                     markerfacecolor="0.70", markeredgecolor="black", markersize=8, label=f"{label}")
    for label in sorted(df_top["dataset_short"].unique())
]
leg1 = ax.legend(handles=handles, frameon=False, title="dataframe", loc="upper left")
ax.add_artist(leg1)

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r"$a$ (\AA)")

ax.set_xlabel(r"$m^{-1/2}$ (amu$^{-1/2}$)")
ax.set_ylabel(r"$\omega_{\max}$ (THz)")
ax.set_title(r"Consistency across independent MOGA runs")
ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
ax.minorticks_on()

fig.tight_layout()
save_figure(fig, "omega_max_mass_scaling_by_dataset")
plt.show()


## Notes for manuscript interpretation

The figures generated here support two simple checks:

1. The maximum phonon frequency follows the expected harmonic trend with approximately inverse square-root mass scaling.
2. After multiplying by \(\sqrt{m}\), the remaining variation is associated primarily with lattice parameter and the selected force-constant families rather than with mass alone.

All figures are saved as `.pdf`, `.png`, and `.svg` files in `mass_lattice_dispersion_publication_figures/`.
